## CPSC 4970 Homework 5 - Jonathan Braun

In [1]:
import numpy as np
import pandas as pd

from pathlib import Path

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import make_scorer

from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.neural_network import MLPClassifier

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

pd.set_option("display.max_colwidth", 120)

## Load and label the data

In [2]:
def load_headlines(path):
    path = Path(path)
    with path.open("r", encoding="utf-8", errors="ignore") as file:
        return [line.strip() for line in file if line.strip()]

clickbait = load_headlines("clickbait_data")
non_clickbait = load_headlines("non_clickbait_data")

print(f"Clickbait headlines: {len(clickbait):,}")
print(f"Non-clickbait headlines: {len(non_clickbait):,}")
print(f"Total headlines: {len(clickbait) + len(non_clickbait):,}")

Clickbait headlines: 15,999
Non-clickbait headlines: 16,001
Total headlines: 32,000


## Create feature list and target vector

In [3]:
X = clickbait + non_clickbait
y = np.array([1] * len(clickbait) + [0] * len(non_clickbait))

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print(f"Training examples: {len(X_train):,}")
print(f"Testing examples: {len(X_test):,}")
print(f"Training class balance: {np.bincount(y_train)}")
print(f"Testing class balance: {np.bincount(y_test)}")

Training examples: 25,600
Testing examples: 6,400
Training class balance: [12801 12799]
Testing class balance: [3200 3200]


## Feature representation and cross-validation setup

In [4]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scoring = {
    "accuracy": "accuracy",
    "precision": make_scorer(precision_score, zero_division=0),
    "recall": make_scorer(recall_score, zero_division=0),
    "f1": make_scorer(f1_score, zero_division=0)
}

tfidf = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    max_features=10000,
    ngram_range=(1, 2)
)

## Model 1: k-nearest neighbors

In [5]:
knn_pipeline = Pipeline([
    ("tfidf", tfidf),
    ("classifier", KNeighborsClassifier())
])

knn_params = {
    "classifier__n_neighbors": [3, 5, 7, 11, 15, 21, 31],
    "classifier__weights": ["uniform", "distance"]
}

knn_grid = GridSearchCV(
    knn_pipeline,
    knn_params,
    cv=cv,
    scoring=scoring,
    refit="f1",
    n_jobs=-1,
    return_train_score=False
)

knn_grid.fit(X_train, y_train)

print("Best kNN parameters:")
print(knn_grid.best_params_)
print(f"Best kNN CV F1 score: {knn_grid.best_score_:.4f}")

Best kNN parameters:
{'classifier__n_neighbors': 3, 'classifier__weights': 'distance'}
Best kNN CV F1 score: 0.7106


## Model 2: Naive Bayes

In [6]:
nb_pipeline = Pipeline([
    ("tfidf", tfidf),
    ("classifier", MultinomialNB())
])

nb_params = {
    "classifier__alpha": [0.01, 0.05, 0.1, 0.5, 1.0, 2.0]
}

nb_grid = GridSearchCV(
    nb_pipeline,
    nb_params,
    cv=cv,
    scoring=scoring,
    refit="f1",
    n_jobs=-1,
    return_train_score=False
)

nb_grid.fit(X_train, y_train)

print("Best Naive Bayes parameters:")
print(nb_grid.best_params_)
print(f"Best Naive Bayes CV F1 score: {nb_grid.best_score_:.4f}")

Best Naive Bayes parameters:
{'classifier__alpha': 1.0}
Best Naive Bayes CV F1 score: 0.9566


## Model 3: Multilayer perceptron

In [7]:
mlp_pipeline = Pipeline([
    ("tfidf", tfidf),
    ("classifier", MLPClassifier(
        max_iter=100,
        early_stopping=True,
        random_state=42
    ))
])

mlp_params = {
    "classifier__hidden_layer_sizes": [
        (25,),
        (50,),
        (100,),
        (25, 25),
        (50, 25),
        (50, 50),
        (100, 50),
        (100, 100),
        (75, 25),
        (100, 50, 25)
    ]
}

mlp_grid = GridSearchCV(
    mlp_pipeline,
    mlp_params,
    cv=cv,
    scoring=scoring,
    refit="f1",
    n_jobs=-1,
    return_train_score=False
)

mlp_grid.fit(X_train, y_train)

print("Best MLP parameters:")
print(mlp_grid.best_params_)
print(f"Best MLP CV F1 score: {mlp_grid.best_score_:.4f}")

Best MLP parameters:
{'classifier__hidden_layer_sizes': (25,)}
Best MLP CV F1 score: 0.9572


## 7. Cross-validation and test-set comparison

After selecting the best version of each model through cross-validation, I compare the models using both cross-validation scores and held-out test-set scores. The cross-validation scores show the mean and standard deviation across the 5 folds, while the test-set scores show how the selected models performed on data that was not used during training or cross-validation.

In [ ]:
models = {
    "k-nearest neighbors": knn_grid,
    "Naive Bayes": nb_grid,
    "Multilayer perceptron": mlp_grid
}

results = []

for model_name, grid in models.items():
    best_index = grid.best_index_
    y_pred = grid.predict(X_test)

    results.append({
        "Model": model_name,
        "Best Parameters": grid.best_params_,
        "CV Accuracy": f"{grid.cv_results_['mean_test_accuracy'][best_index]:.4f} +/- {grid.cv_results_['std_test_accuracy'][best_index]:.4f}",
        "CV Precision": f"{grid.cv_results_['mean_test_precision'][best_index]:.4f} +/- {grid.cv_results_['std_test_precision'][best_index]:.4f}",
        "CV Recall": f"{grid.cv_results_['mean_test_recall'][best_index]:.4f} +/- {grid.cv_results_['std_test_recall'][best_index]:.4f}",
        "CV F1": f"{grid.cv_results_['mean_test_f1'][best_index]:.4f} +/- {grid.cv_results_['std_test_f1'][best_index]:.4f}",
        "Test Accuracy": accuracy_score(y_test, y_pred),
        "Test Precision": precision_score(y_test, y_pred, zero_division=0),
        "Test Recall": recall_score(y_test, y_pred, zero_division=0),
        "Test F1": f1_score(y_test, y_pred, zero_division=0)
    })

results_df = pd.DataFrame(results)
results_df

In [ ]:
best_model_name = results_df.sort_values("Test F1", ascending=False).iloc[0]["Model"]
best_model_name